# Thực nghiệm: Ảnh hưởng của kích thước Latent Space đến hiệu năng phát hiện bất thường của VAE

**Dự án:** NIDS-VAE — Phát hiện xâm nhập mạng bằng Variational Autoencoder  
**Dataset:** CICIDS2017  
**Mục tiêu:** So sánh 3 cấu hình `latent_dim` ∈ {8, 16, 32} để chứng minh tại sao chọn `latent_dim=16`

---

> **Cách chạy:**
> - Đặt `QUICK_EXPERIMENT = True` để chạy nhanh (5 epoch, subset 10k mẫu) — dùng để kiểm tra
> - Đặt `QUICK_EXPERIMENT = False` để chạy đầy đủ (100 epoch, toàn bộ dữ liệu) — dùng cho báo cáo

In [22]:
# ============================================================
# CẤU HÌNH THÍ NGHIỆM
# ------------------------------------------------------------
# QUICK_EXPERIMENT = True  : chạy nhanh để kiểm tra notebook
# QUICK_EXPERIMENT = False : chạy đầy đủ để lấy kết quả báo cáo
# ============================================================
QUICK_EXPERIMENT = False

# ── Import thư viện chuẩn ────────────────────────────────────
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Không hiển thị cửa sổ GUI — lưu file trực tiếp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')

# ── Thiết lập đường dẫn gốc project ─────────────────────────
# notebooks/ nằm trong PROJECT_ROOT, nên cần lùi 1 cấp
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Import VAE từ backend (nguồn duy nhất của kiến trúc) ─────
from backend.app.models.vae import VAE, vae_loss  # noqa: E402

# ── Đường dẫn dữ liệu ────────────────────────────────────────
DATA_TRAIN_DIR = PROJECT_ROOT / 'data' / 'train'
DATA_VAL_DIR   = PROJECT_ROOT / 'data' / 'validation'
DATA_TEST_DIR  = PROJECT_ROOT / 'data' / 'test'

# ── Thư mục lưu kết quả thí nghiệm ──────────────────────────
EXPERIMENTS_DIR = PROJECT_ROOT / 'artifacts' / 'experiments'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT     : {PROJECT_ROOT}')
print(f'EXPERIMENTS_DIR  : {EXPERIMENTS_DIR}')
print(f'QUICK_EXPERIMENT : {QUICK_EXPERIMENT}')
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')

PROJECT_ROOT     : D:\nids-vae-project
EXPERIMENTS_DIR  : D:\nids-vae-project\artifacts\experiments
QUICK_EXPERIMENT : False
PyTorch version  : 2.12.0+cpu
CUDA available   : False


---
## SECTION 1 — Giới thiệu thí nghiệm

### 1.1 Latent Space trong VAE là gì?

Trong Variational Autoencoder (VAE), **latent space** (không gian tiềm ẩn) là tầng nén trung gian nơi dữ liệu đầu vào được ánh xạ thành một phân phối xác suất $q(z|x) = \mathcal{N}(\mu, \sigma^2)$.

```
Input x (66 features)
    ↓ Encoder
  [128] → [64] → (μ, σ²) ∈ ℝ^latent_dim
    ↓ Reparameterization: z = μ + ε·σ
  z ∈ ℝ^latent_dim
    ↓ Decoder
  [64] → [128] → x̂ (66 features)
```

**Reconstruction error** được tính là:
$$\text{error}_i = \frac{1}{d} \sum_{j=1}^{d} (x_{ij} - \hat{x}_{ij})^2$$

### 1.2 Vai trò của `latent_dim`

| Trường hợp | Hậu quả |
|---|---|
| `latent_dim` quá nhỏ | Mất thông tin quan trọng, decoder khó tái tạo → recall thấp |
| `latent_dim` quá lớn | Giữ cả nhiễu, mô hình tái tạo tốt cả anomaly → FPR tăng, ROC-AUC giảm |
| `latent_dim` cân bằng | Nén đủ để học cấu trúc bình thường, tái tạo kém anomaly → phân biệt tốt |

### 1.3 Giả thuyết nghiên cứu

**H1:** `latent_dim=8` sẽ cho Recall thấp hơn do mất thông tin quan trọng khi nén mạnh.  
**H2:** `latent_dim=32` sẽ cho FPR cao hơn do không gian tiềm ẩn lớn giữ lại nhiễu.  
**H3:** `latent_dim=16` đạt điểm cân bằng tốt nhất giữa khả năng biểu diễn và khả năng phân biệt anomaly.

---
## SECTION 2 — Load dữ liệu

In [23]:
# ============================================================
# TẢI DỮ LIỆU ĐÃ TIỀN XỬ LÝ
# ------------------------------------------------------------
# Dữ liệu được tạo bởi scripts/clean_data.py:
#   - X_*.npy : features đã chuẩn hóa (StandardScaler)
#   - y_*.npy : nhãn nhị phân (0=BENIGN, 1=ATTACK)
# ============================================================

print('Đang tải dữ liệu...')

X_train = np.load(DATA_TRAIN_DIR / 'X_train.npy').astype(np.float32)
y_train = np.load(DATA_TRAIN_DIR / 'y_train.npy').astype(np.int32)

X_val   = np.load(DATA_VAL_DIR / 'X_val.npy').astype(np.float32)
y_val   = np.load(DATA_VAL_DIR / 'y_val.npy').astype(np.int32)

X_test  = np.load(DATA_TEST_DIR / 'X_test.npy').astype(np.float32)
y_test  = np.load(DATA_TEST_DIR / 'y_test.npy').astype(np.int32)

# ── Xác nhận không có NaN/Inf trước khi dùng ─────────────────
for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    assert not np.isnan(arr).any(), f'{name} chứa NaN'
    assert not np.isinf(arr).any(), f'{name} chứa Inf'

print('Kiểm tra NaN/Inf: OK')

# ── Hiển thị thông tin shape ──────────────────────────────────
print(f'\n{"Tập dữ liệu":<12} {"Shape":<22} {"BENIGN":<12} {"ATTACK":<12} {"Attack rate"}')
print('-' * 70)
for name, X, y in [('Train', X_train, y_train),
                    ('Validation', X_val,   y_val),
                    ('Test',  X_test,  y_test)]:
    n_benign = int((y == 0).sum())
    n_attack = int((y == 1).sum())
    rate = n_attack / len(y) * 100
    print(f'{name:<12} {str(X.shape):<22} {n_benign:<12,d} {n_attack:<12,d} {rate:.1f}%')

# ── Lưu lại input_dim để dùng trong suốt notebook ────────────
INPUT_DIM = X_train.shape[1]
print(f'\nInput dimension : {INPUT_DIM}')

# ── Chế độ QUICK_EXPERIMENT: dùng subset nhỏ để test nhanh ──
if QUICK_EXPERIMENT:
    X_train_exp = X_train[:10_000]
    X_val_exp   = X_val[:min(2000, len(X_val))]
    y_val_exp   = y_val[:min(2000, len(y_val))]

    # Chọn test subset đảm bảo có cả 2 lớp (BENIGN + ATTACK) để tính ROC-AUC
    # Lấy 2500 mẫu BENIGN đầu + 2500 mẫu ATTACK đầu, sau đó xáo trộn
    idx_benign = np.where(y_test == 0)[0][:2500]
    idx_attack = np.where(y_test == 1)[0][:2500]
    idx_quick  = np.concatenate([idx_benign, idx_attack])
    rng = np.random.default_rng(seed=42)
    rng.shuffle(idx_quick)
    X_test_exp = X_test[idx_quick]
    y_test_exp = y_test[idx_quick]

    n_q_benign = int((y_test_exp == 0).sum())
    n_q_attack = int((y_test_exp == 1).sum())
    print(f'\n[QUICK] train={len(X_train_exp):,} | val={len(X_val_exp):,} | '
          f'test={len(X_test_exp):,} ({n_q_benign} BENIGN + {n_q_attack} ATTACK)')
else:
    # Chạy đầy đủ: dùng toàn bộ dữ liệu
    X_train_exp = X_train
    X_val_exp   = X_val
    X_test_exp  = X_test
    y_test_exp  = y_test
    y_val_exp   = y_val
    print(f'\n[FULL] train={len(X_train_exp):,} | val={len(X_val_exp):,} | test={len(X_test_exp):,}')


Đang tải dữ liệu...
Kiểm tra NaN/Inf: OK

Tập dữ liệu  Shape                  BENIGN       ATTACK       Attack rate
----------------------------------------------------------------------
Train        (402229, 66)           402,229      0            0.0%
Validation   (100558, 66)           100,558      0            0.0%
Test         (2019575, 66)          1,593,697    425,878      21.1%

Input dimension : 66

[FULL] train=402,229 | val=100,558 | test=2,019,575


---
## SECTION 3 — Hàm hỗ trợ & Cấu hình thí nghiệm

Các hàm dùng lại xuyên suốt notebook: `set_seed`, `get_beta`, `train_epoch`, `eval_epoch`, `compute_reconstruction_errors`, `select_threshold_percentile`, `compute_metrics`, `save_experiment_artifacts`.

Cấu hình hyperparameters chung — chỉ thay đổi `latent_dim`, tất cả tham số khác giữ nguyên để kết quả có thể so sánh công bằng.

In [24]:
# ============================================================
# HÀM HỖ TRỢ HUẤN LUYỆN
# ------------------------------------------------------------
# Các hàm này được viết độc lập để notebook chạy được
# mà không phụ thuộc vào scripts/train.py.
# Logic giống hệt train.py để đảm bảo tính nhất quán.
# ============================================================

def set_seed(seed: int) -> None:
    """Đặt seed toàn cục để kết quả có thể tái tạo (reproducible)."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_beta(epoch: int, beta_start: float, beta_end: float,
             warmup_epochs: int) -> float:
    """Tính beta động theo lịch KL Annealing tuyến tính.
    
    Mục đích: tránh posterior collapse bằng cách bắt đầu beta=0
    (chỉ học reconstruction), sau đó tăng dần đến beta_end.
    """
    if warmup_epochs <= 0:
        return beta_end
    progress = min(float(epoch) / float(warmup_epochs), 1.0)
    return beta_start + (beta_end - beta_start) * progress


def train_epoch(model: VAE, loader: DataLoader, optimizer,
                device: torch.device, beta: float,
                grad_clip: float) -> dict:
    """Huấn luyện 1 epoch, trả về loss trung bình (total, recon, kl)."""
    model.train()
    total_sum = recon_sum = kl_sum = 0.0
    n_batches = 0

    for (batch_x,) in loader:
        batch_x = batch_x.to(device, non_blocking=True)
        optimizer.zero_grad()
        x_hat, mu, logvar = model(batch_x)
        loss, recon, kl = vae_loss(batch_x, x_hat, mu, logvar, beta=beta)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()
        total_sum += loss.item()
        recon_sum += recon.item()
        kl_sum    += kl.item()
        n_batches += 1

    return {'loss': total_sum / n_batches,
            'recon': recon_sum / n_batches,
            'kl': kl_sum / n_batches}


@torch.no_grad()
def eval_epoch(model: VAE, loader: DataLoader,
               device: torch.device, beta: float) -> dict:
    """Đánh giá model trên val set mà không cập nhật gradient."""
    model.eval()
    total_sum = recon_sum = kl_sum = 0.0
    n_batches = 0

    for (batch_x,) in loader:
        batch_x = batch_x.to(device, non_blocking=True)
        x_hat, mu, logvar = model(batch_x)
        loss, recon, kl = vae_loss(batch_x, x_hat, mu, logvar, beta=beta)
        total_sum += loss.item()
        recon_sum += recon.item()
        kl_sum    += kl.item()
        n_batches += 1

    return {'loss': total_sum / n_batches,
            'recon': recon_sum / n_batches,
            'kl': kl_sum / n_batches}


@torch.no_grad()
def compute_reconstruction_errors(model: VAE, X: np.ndarray,
                                   batch_size: int,
                                   device: torch.device) -> np.ndarray:
    """Tính MSE reconstruction error cho từng mẫu.
    
    Công thức nhất quán với backend inference:
        error_i = mean((x_i - x̂_i)²) theo chiều feature
    """
    model.eval()
    tensor_X = torch.tensor(X, dtype=torch.float32)
    loader   = DataLoader(TensorDataset(tensor_X),
                          batch_size=batch_size, shuffle=False, num_workers=0)
    all_errors = []
    for (batch,) in loader:
        batch = batch.to(device)
        x_hat, _, _ = model(batch)
        # Dùng static method từ VAE class — cùng formula với backend
        errors = VAE.reconstruction_error(batch, x_hat)
        all_errors.append(errors.cpu().numpy())
    return np.concatenate(all_errors, axis=0)


def select_threshold_percentile(val_errors: np.ndarray,
                                  percentile: float = 99.0) -> float:
    """Chọn ngưỡng anomaly bằng phương pháp percentile trên validation errors.
    
    Dùng val set (chỉ BENIGN) để không rò rỉ thông tin từ test set.
    """
    return float(np.percentile(val_errors, percentile))


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                    test_errors: np.ndarray) -> dict:
    """Tính đầy đủ các metrics phát hiện anomaly."""
    from sklearn.metrics import confusion_matrix
    acc       = float(accuracy_score(y_true, y_pred))
    precision = float(precision_score(y_true, y_pred, zero_division=0))
    recall    = float(recall_score(y_true, y_pred, zero_division=0))
    f1        = float(f1_score(y_true, y_pred, zero_division=0))

    # ROC-AUC dùng reconstruction error liên tục thay vì nhị phân
    try:
        roc_auc = float(roc_auc_score(y_true, test_errors))
    except ValueError:
        roc_auc = float('nan')

    # FPR từ confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0
    else:
        fpr = float('nan')

    return {
        'accuracy':            acc,
        'precision':           precision,
        'recall':              recall,
        'f1_score':            f1,
        'roc_auc':             roc_auc,
        'false_positive_rate': fpr,
    }


def save_experiment_artifacts(exp_dir: Path, latent_dim: int,
                               model_config: dict, training_summary: dict,
                               threshold_val: float, eval_metrics: dict,
                               percentile: float = 99.0) -> None:
    """Lưu tất cả artifact của một cấu hình thí nghiệm vào thư mục riêng."""
    exp_dir.mkdir(parents=True, exist_ok=True)

    # model_config.json
    with open(exp_dir / 'model_config.json', 'w', encoding='utf-8') as f:
        json.dump(model_config, f, indent=2, ensure_ascii=False)

    # training_summary.json
    with open(exp_dir / 'training_summary.json', 'w', encoding='utf-8') as f:
        json.dump(training_summary, f, indent=2, ensure_ascii=False)

    # threshold.json
    threshold_artifact = {
        'threshold':        threshold_val,
        'percentile':       percentile,
        'selection_method': 'percentile_on_validation_errors',
    }
    with open(exp_dir / 'threshold.json', 'w', encoding='utf-8') as f:
        json.dump(threshold_artifact, f, indent=2, ensure_ascii=False)

    # evaluation_metrics.json
    with open(exp_dir / 'evaluation_metrics.json', 'w', encoding='utf-8') as f:
        json.dump({'metrics': eval_metrics, 'threshold': threshold_val,
                   'latent_dim': latent_dim}, f, indent=2, ensure_ascii=False)

    print(f'  Đã lưu artifacts: {exp_dir}')


print('Đã định nghĩa các hàm hỗ trợ huấn luyện OK')

Đã định nghĩa các hàm hỗ trợ huấn luyện OK


In [25]:
# ============================================================
# THAM SỐ CHUNG — giữ nguyên cho cả 3 cấu hình
# ------------------------------------------------------------
# Tuân theo nguyên tắc: chỉ thay đổi latent_dim,
# giữ nguyên tất cả tham số khác để kết quả có thể so sánh
# ============================================================

HIDDEN_DIMS    = [128, 64]   # Kiến trúc encoder/decoder
LEARNING_RATE  = 1e-3        # Adam optimizer learning rate
WEIGHT_DECAY   = 1e-5        # L2 regularization
BATCH_SIZE     = 1024        # Kích thước batch
GRAD_CLIP      = 5.0         # Ngưỡng gradient clipping
RANDOM_SEED    = 42          # Seed cho tái tạo kết quả
THRESHOLD_PCT  = 99.0        # Percentile để chọn ngưỡng anomaly
EVAL_BATCH     = 4096        # Batch size khi tính reconstruction errors
BETA_START     = 0.0         # Beta ban đầu (KL Annealing)
BETA_END       = 1.0         # Beta cuối (standard VAE)

# ── Tham số phụ thuộc QUICK_EXPERIMENT ───────────────────────
if QUICK_EXPERIMENT:
    MAX_EPOCHS     = 5       # Chạy nhanh: 5 epoch
    PATIENCE       = 3       # Early stopping sau 3 epoch
    WARMUP_EPOCHS  = 2       # KL warmup nhanh
    print('[QUICK] MAX_EPOCHS=5, PATIENCE=3, WARMUP=2')
else:
    MAX_EPOCHS     = 100     # Chạy đầy đủ: 100 epoch
    PATIENCE       = 10      # Early stopping sau 10 epoch
    WARMUP_EPOCHS  = 30      # KL warmup chuẩn 30 epoch
    print('[FULL] MAX_EPOCHS=100, PATIENCE=10, WARMUP=30')

# ── Device: ưu tiên CUDA nếu có ──────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── 3 cấu hình thí nghiệm ────────────────────────────────────
LATENT_CONFIGS = [8, 16, 32]
print(f'Latent configs: {LATENT_CONFIGS}')

[FULL] MAX_EPOCHS=100, PATIENCE=10, WARMUP=30
Device: cpu
Latent configs: [8, 16, 32]


---
## SECTION 4 — Warmup-Aware Early Stopping: `train_config_v2()`

### Vì sao cần warmup-aware early stopping?

Trong giai đoạn KL warmup (epoch 1 → 30), beta tăng dần từ 0 → 1. Do đó:

$$\text{val\_loss} = \text{recon} + \beta \cdot \text{kl}$$

tăng tự nhiên mỗi epoch — **dù model không hề xấu đi**. Code v1 nhầm lẫn đây là "không cải thiện" → early stopping kích hoạt tại epoch 11 (= 1 + patience), `best_epoch = 1` sai hoàn toàn.

### Logic `train_config_v2()` — 3 phase

| Phase | Điều kiện | Hành động |
|-------|-----------|-----------|
| **WARMUP** | epoch ≤ 30 | Checkpoint theo `val_recon`. **KHÔNG** đếm patience. |
| **POST_WARMUP_START** | epoch = 31 | Reset `best_val_loss = ∞`, `epochs_no_improve = 0`. |
| **POST_WARMUP** | epoch > 31 | Early stopping bình thường theo `total val_loss`. |

> Tham chiếu: `scripts/train.py` — biến `in_warmup`, `just_entered_post_warmup`

In [33]:
# ============================================================
# ĐỊNH NGHĨA train_config_v2() — WARMUP-AWARE EARLY STOPPING
# ------------------------------------------------------------
# Khắc phục bug trong v1: early stopping kích hoạt trong giai
# đoạn warmup do total val_loss tăng theo beta, không phải do
# model xấu đi.
#
# Logic theo scripts/train.py (biến in_warmup, just_entered_post_warmup):
#   PHASE 1 WARMUP  : Không đếm patience. Checkpoint theo val_recon.
#   PHASE 2 RESET   : Reset toàn bộ trạng thái khi bước vào post-warmup.
#   PHASE 3 POST    : Early stopping bình thường theo total val_loss.
# ============================================================


def train_config_v2(
    latent_dim: int,
    X_train: np.ndarray,
    X_val: np.ndarray,
    output_dir: Path,
    cfg: dict,
) -> dict:
    """
    Huấn luyện VAE với warmup-aware early stopping — phiên bản v2.

    Khắc phục bug best_epoch=1 trong v1: code cũ kích hoạt early
    stopping trong giai đoạn warmup vì total val_loss tăng tự nhiên
    theo beta, không phản ánh model xấu đi.

    PHASE 1 — WARMUP (epoch <= beta_warmup_epochs):
        - Beta đang tăng từ 0 → beta_end → total val_loss tự nhiên tăng.
        - KHÔNG đếm epochs_no_improve (không kích hoạt patience).
        - Checkpoint theo val_recon_loss (ổn định, không phụ thuộc beta).

    PHASE 2 — KẾT THÚC WARMUP (epoch == beta_warmup_epochs + 1):
        - Reset: best_val_loss = ∞, epochs_no_improve = 0, best_epoch = None.
        - Lưu checkpoint đầu tiên post-warmup làm mốc so sánh.
        - Bắt đầu monitor total val_loss với beta đã cố định.

    PHASE 3 — POST-WARMUP (epoch > beta_warmup_epochs):
        - Beta = 1.0 (cố định) → total val_loss phản ánh đúng chất lượng.
        - Early stopping bình thường theo total val_loss.
        - Checkpoint khi total val_loss giảm.

    Args:
        latent_dim : Số chiều không gian tiềm ẩn cần thí nghiệm.
        X_train    : Mảng features huấn luyện float32 (chỉ BENIGN flows).
        X_val      : Mảng features validation float32 (chỉ BENIGN flows).
        output_dir : Thư mục lưu checkpoint và artifacts của cấu hình này.
        cfg        : Dict chứa toàn bộ hyperparameters.

    Returns:
        Dict kết quả gồm: model, ckpt_path, history, best_epoch,
        final_epoch, best_val_loss, stopped_early, n_params, elapsed_s.
    """
    # ── Đặt seed để kết quả có thể tái tạo ────────────────────
    set_seed(cfg['seed'])
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = output_dir / 'vae_best.pth'

    # ── Khởi tạo VAE với latent_dim được chỉ định ──────────────
    model = VAE(
        input_dim   = cfg['input_dim'],
        latent_dim  = latent_dim,
        hidden_dims = cfg['hidden_dims'],
    ).to(cfg['device'])
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Số tham số   : {n_params:,}')

    # ── Tạo DataLoader cho train và val ────────────────────────
    pin = (cfg['device'].type == 'cuda')
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train)),
        batch_size=cfg['batch_size'], shuffle=True,
        num_workers=0, pin_memory=pin,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_val)),
        batch_size=cfg['batch_size'], shuffle=False,
        num_workers=0, pin_memory=pin,
    )

    # ── Optimizer: Adam với L2 regularization ──────────────────
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
    )

    # ── Trạng thái warmup-aware early stopping ─────────────────
    # best_val_loss  : theo dõi val_recon trong warmup,
    #                  total val_loss trong post-warmup.
    # epochs_no_improve: chỉ đếm trong post-warmup.
    # best_epoch     : None cho đến khi bước vào post-warmup.
    best_val_loss     = float('inf')
    epochs_no_improve = 0
    best_epoch        = None

    # Lịch sử training: gồm loss components + beta + phase hiện tại
    history = {
        'train_loss' : [], 'val_loss'  : [],
        'train_recon': [], 'val_recon' : [],
        'train_kl'   : [], 'val_kl'    : [],
        'beta'       : [], 'phase'     : [],
    }

    start_time    = time.time()
    stopped_early = False
    final_epoch   = cfg['max_epochs']
    beta_warmup   = cfg['beta_warmup_epochs']

    for epoch in range(1, cfg['max_epochs'] + 1):

        # ── Tính beta theo lịch KL Annealing tuyến tính ────────
        beta = get_beta(epoch, cfg['beta_start'], cfg['beta_end'], beta_warmup)

        # ── Xác định phase của epoch hiện tại ──────────────────
        in_warmup               = (epoch <= beta_warmup)
        just_entered_post_warmup = (epoch == beta_warmup + 1)

        if epoch < beta_warmup:
            phase = 'WARMUP'
        elif epoch == beta_warmup:
            phase = 'WARMUP_END'
        elif just_entered_post_warmup:
            phase = 'POST_WARMUP_START'
        else:
            phase = 'POST_WARMUP'

        # ── Huấn luyện + đánh giá một epoch ────────────────────
        train_stats = train_epoch(
            model, train_loader, optimizer,
            cfg['device'], beta, cfg['grad_clip'],
        )
        val_stats = eval_epoch(model, val_loader, cfg['device'], beta)

        # ── Ghi lịch sử loss và thông tin phase ────────────────
        history['train_loss'].append(train_stats['loss'])
        history['val_loss'].append(val_stats['loss'])
        history['train_recon'].append(train_stats['recon'])
        history['val_recon'].append(val_stats['recon'])
        history['train_kl'].append(train_stats['kl'])
        history['val_kl'].append(val_stats['kl'])
        history['beta'].append(beta)
        history['phase'].append(phase)

        # ── Log từng epoch rõ phase ────────────────────────────
        print(
            f'  Epoch {epoch:03d} | beta={beta:.4f} | phase={phase:<20} | '
            f'val_loss={val_stats["loss"]:.4f} | '
            f'val_recon={val_stats["recon"]:.4f}'
        )

        # ── Logic warmup-aware early stopping ──────────────────
        if just_entered_post_warmup:
            # Reset toàn bộ: bắt đầu so sánh total val_loss
            # với beta đã cố định = beta_end.
            best_val_loss     = float('inf')
            epochs_no_improve = 0
            best_epoch        = None

            # Lưu checkpoint đầu tiên post-warmup làm mốc
            torch.save(model.state_dict(), ckpt_path)
            best_val_loss = val_stats['loss']
            best_epoch    = epoch
            print(
                f'  → [POST_WARMUP] Reset early stopping | '
                f'best_epoch={epoch} | val_loss={val_stats["loss"]:.6f}'
            )

        elif in_warmup:
            # WARMUP: checkpoint theo val_recon, KHÔNG đếm patience.
            # val_recon không bị ảnh hưởng bởi beta tăng → ổn định hơn.
            if val_stats['recon'] < best_val_loss:
                best_val_loss = val_stats['recon']
                best_epoch    = epoch
                torch.save(model.state_dict(), ckpt_path)
                print(
                    f'  → [WARMUP] Checkpoint lưu '
                    f'(val_recon={val_stats["recon"]:.6f})'
                )
            else:
                print(
                    f'  → [WARMUP] beta={beta:.4f} | '
                    f'val_recon={val_stats["recon"]:.6f} | patience: N/A'
                )

        else:
            # POST_WARMUP: early stopping bình thường theo total val_loss.
            # Beta đã = beta_end (không đổi) → val_loss ổn định để so sánh.
            if val_stats['loss'] < best_val_loss:
                best_val_loss     = val_stats['loss']
                epochs_no_improve = 0
                best_epoch        = epoch
                torch.save(model.state_dict(), ckpt_path)
                print(
                    f'  → [POST_WARMUP] Best model '
                    f'(val_loss={val_stats["loss"]:.6f}) | best_epoch={epoch}'
                )
            else:
                epochs_no_improve += 1
                print(
                    f'  → [POST_WARMUP] Không cải thiện '
                    f'{epochs_no_improve}/{cfg["patience"]}'
                )
                if epochs_no_improve >= cfg['patience']:
                    stopped_early = True
                    final_epoch   = epoch
                    print(
                        f'  → Early stopping kích hoạt tại epoch {epoch} '
                        f'(best_epoch={best_epoch})'
                    )
                    break

    elapsed = time.time() - start_time
    # Nếu không stopped_early thì final_epoch = epoch cuối của vòng lặp
    if not stopped_early:
        final_epoch = epoch  # noqa

    return {
        'model'        : model,
        'ckpt_path'    : ckpt_path,
        'history'      : history,
        'best_epoch'   : best_epoch,
        'final_epoch'  : final_epoch,
        'best_val_loss': best_val_loss,
        'stopped_early': stopped_early,
        'n_params'     : n_params,
        'elapsed_s'    : elapsed,
    }


print('Đã định nghĩa train_config_v2() — warmup-aware early stopping ✓')

Đã định nghĩa train_config_v2() — warmup-aware early stopping ✓


---
## SECTION 5 — Chạy thí nghiệm v2

Huấn luyện 3 cấu hình `latent_dim ∈ {8, 16, 32}` với `train_config_v2()` (warmup-aware early stopping).  
Kết quả lưu tại `artifacts/experiments/latent_dimension_v2/` — **không ghi đè kết quả v1 cũ**.

In [34]:
# ============================================================
# CẤU HÌNH v2 — HYPERPARAMETERS VÀ THƯ MỤC OUTPUT
# ------------------------------------------------------------
# Thư mục riêng để KHÔNG ghi đè kết quả v1.
# Giữ nguyên tất cả hyperparameters từ Section 3 để so sánh
# công bằng: chỉ fix logic early stopping.
# ============================================================

# Thư mục output mới — hoàn toàn tách biệt khỏi kết quả cũ
V2_EXPERIMENTS_DIR = PROJECT_ROOT / 'artifacts' / 'experiments' / 'latent_dimension_v2'
V2_EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'V2_EXPERIMENTS_DIR : {V2_EXPERIMENTS_DIR}')

# ── Cấu hình hyperparameters — giữ nguyên từ Section 3 ───────
# Tất cả giá trị đều lấy từ biến đã khai báo ở Section 3,
# không hardcode để đảm bảo nhất quán.
V2_CFG = {
    'input_dim'         : INPUT_DIM,
    'hidden_dims'       : HIDDEN_DIMS,
    'lr'                : LEARNING_RATE,
    'weight_decay'      : WEIGHT_DECAY,
    'batch_size'        : BATCH_SIZE,
    'grad_clip'         : GRAD_CLIP,
    'seed'              : RANDOM_SEED,
    'max_epochs'        : MAX_EPOCHS,
    'patience'          : PATIENCE,
    'beta_warmup_epochs': WARMUP_EPOCHS,
    'beta_start'        : BETA_START,
    'beta_end'          : BETA_END,
    'device'            : DEVICE,
}

print('\nHyperparameters v2 (giữ nguyên từ Section 3):')
for k, v in V2_CFG.items():
    if k != 'device':
        print(f'  {k:<22} : {v}')
print(f'  {"device":<22} : {DEVICE}')

if QUICK_EXPERIMENT:
    print('\n' + '!' * 60)
    print('  ⚠️  QUICK_EXPERIMENT = True')
    print('  Quick mode KHÔNG dùng cho báo cáo chính thức.')
    print('  Đặt QUICK_EXPERIMENT=False và chạy lại toàn bộ notebook.')
    print('!' * 60)
else:
    print('\n✓ QUICK_EXPERIMENT = False — kết quả đầy đủ cho báo cáo.')

V2_EXPERIMENTS_DIR : D:\nids-vae-project\artifacts\experiments\latent_dimension_v2

Hyperparameters v2 (giữ nguyên từ Section 3):
  input_dim              : 66
  hidden_dims            : [128, 64]
  lr                     : 0.001
  weight_decay           : 1e-05
  batch_size             : 1024
  grad_clip              : 5.0
  seed                   : 42
  max_epochs             : 100
  patience               : 10
  beta_warmup_epochs     : 30
  beta_start             : 0.0
  beta_end               : 1.0
  device                 : cpu

✓ QUICK_EXPERIMENT = False — kết quả đầy đủ cho báo cáo.


In [35]:
# ============================================================
# VÒNG LẶP THÍ NGHIỆM v2 — 3 CẤU HÌNH LATENT DIMENSION
# ------------------------------------------------------------
# Mỗi cấu hình được huấn luyện với train_config_v2():
#   - Warmup-aware early stopping (bug fixed từ v1)
#   - Checkpoint theo val_recon trong warmup
#   - Monitor total val_loss trong post-warmup
#
# Artifacts lưu tại: artifacts/experiments/latent_dimension_v2/
# ============================================================

# Lưu kết quả tất cả cấu hình để so sánh
all_results_v2 = {}   # latent_dim -> dict kết quả

for latent_dim in LATENT_CONFIGS:
    print(f'\n{"=" * 65}')
    print(f'  EXPERIMENT v2: latent_dim = {latent_dim}')
    print(f'{"=" * 65}')

    # ── Thư mục output riêng cho từng cấu hình ─────────────────
    exp_dir_v2 = V2_EXPERIMENTS_DIR / f'latent_{latent_dim}'

    # ── Huấn luyện với warmup-aware early stopping ─────────────
    result = train_config_v2(
        latent_dim = latent_dim,
        X_train    = X_train_exp,
        X_val      = X_val_exp,
        output_dir = exp_dir_v2,
        cfg        = V2_CFG,
    )

    print(f'\n  Kết quả: best_epoch={result["best_epoch"]} | '
          f'final_epoch={result["final_epoch"]} | '
          f'best_val_loss={result["best_val_loss"]:.6f} | '
          f'time={result["elapsed_s"]:.1f}s')

    # ── Tải lại checkpoint tốt nhất để đánh giá ────────────────
    model = result['model']
    model.load_state_dict(
        torch.load(result['ckpt_path'], map_location=DEVICE, weights_only=True)
    )
    model.eval()

    # ── Tính reconstruction errors trên val và test ─────────────
    print('  Đang tính reconstruction errors...')
    val_errors  = compute_reconstruction_errors(
        model, X_val_exp, EVAL_BATCH, DEVICE
    )
    test_errors = compute_reconstruction_errors(
        model, X_test_exp, EVAL_BATCH, DEVICE
    )

    # ── Chọn ngưỡng anomaly từ validation errors ────────────────
    # Dùng percentile trên val set (chỉ BENIGN) — tránh data leakage
    threshold = select_threshold_percentile(val_errors, THRESHOLD_PCT)
    print(f'  Ngưỡng anomaly (p{THRESHOLD_PCT:.0f}) : {threshold:.4f}')

    # ── Dự đoán nhãn và tính metrics ───────────────────────────
    y_pred  = (test_errors > threshold).astype(np.int32)
    metrics = compute_metrics(y_test_exp, y_pred, test_errors)
    print(
        f'  Precision={metrics["precision"]:.4f} | '
        f'Recall={metrics["recall"]:.4f} | '
        f'F1={metrics["f1_score"]:.4f} | '
        f'ROC-AUC={metrics["roc_auc"]:.4f} | '
        f'FPR={metrics["false_positive_rate"]:.4f}'
    )

    # ── Lưu training_history.json ───────────────────────────────
    with open(exp_dir_v2 / 'training_history.json', 'w', encoding='utf-8') as fh:
        json.dump(result['history'], fh, indent=2, ensure_ascii=False)

    # ── Chuẩn bị artifacts ─────────────────────────────────────
    model_config_v2 = {
        'architecture'     : 'VAE',
        'input_dim'        : INPUT_DIM,
        'latent_dim'       : latent_dim,
        'hidden_dims'      : HIDDEN_DIMS,
        'beta'             : BETA_END,
        'activation'       : 'ReLU',
        'output_activation': 'Linear',
        'training_version' : 'v2_warmup_aware',
    }
    training_summary_v2 = {
        'hyperparameters': {
            'input_dim'         : INPUT_DIM,
            'latent_dim'        : latent_dim,
            'hidden_dims'       : HIDDEN_DIMS,
            'beta'              : BETA_END,
            'learning_rate'     : LEARNING_RATE,
            'weight_decay'      : WEIGHT_DECAY,
            'batch_size'        : BATCH_SIZE,
            'max_epochs'        : MAX_EPOCHS,
            'patience'          : PATIENCE,
            'grad_clip'         : GRAD_CLIP,
            'seed'              : RANDOM_SEED,
            'beta_warmup_epochs': WARMUP_EPOCHS,
            'beta_start'        : BETA_START,
            'beta_end'          : BETA_END,
        },
        'results': {
            'best_epoch'   : result['best_epoch'],
            'final_epoch'  : result['final_epoch'],
            'stopped_early': result['stopped_early'],
            'best_val_loss': result['best_val_loss'],
            'n_params'     : result['n_params'],
        },
        'fix_note'            : 'v2: warmup-aware early stopping — bug fixed from v1',
        'total_time_seconds'  : round(result['elapsed_s'], 2),
    }

    # ── Lưu tất cả artifacts qua helper function ────────────────
    save_experiment_artifacts(
        exp_dir          = exp_dir_v2,
        latent_dim       = latent_dim,
        model_config     = model_config_v2,
        training_summary = training_summary_v2,
        threshold_val    = threshold,
        eval_metrics     = metrics,
        percentile       = THRESHOLD_PCT,
    )

    # ── Ghi vào dict tổng hợp ───────────────────────────────────
    all_results_v2[latent_dim] = {
        'latent_dim'   : latent_dim,
        'n_params'     : result['n_params'],
        'best_epoch'   : result['best_epoch'],
        'final_epoch'  : result['final_epoch'],
        'best_val_loss': result['best_val_loss'],
        'threshold'    : threshold,
        'metrics'      : metrics,
        'history'      : result['history'],
        'elapsed_s'    : result['elapsed_s'],
        'stopped_early': result['stopped_early'],
    }

print(f'\n{"=" * 65}')
print('Hoàn thành tất cả 3 cấu hình v2 (warmup-aware early stopping)!')
print(f'{"=" * 65}')


  EXPERIMENT v2: latent_dim = 8
  Số tham số   : 35,282
  Epoch 001 | beta=0.0333 | phase=WARMUP               | val_loss=0.3797 | val_recon=0.2666
  → [WARMUP] Checkpoint lưu (val_recon=0.266580)
  Epoch 002 | beta=0.0667 | phase=WARMUP               | val_loss=0.4016 | val_recon=0.2320
  → [WARMUP] Checkpoint lưu (val_recon=0.231953)
  Epoch 003 | beta=0.1000 | phase=WARMUP               | val_loss=0.4332 | val_recon=0.2268
  → [WARMUP] Checkpoint lưu (val_recon=0.226830)
  Epoch 004 | beta=0.1333 | phase=WARMUP               | val_loss=0.4419 | val_recon=0.2302
  → [WARMUP] beta=0.1333 | val_recon=0.230224 | patience: N/A
  Epoch 005 | beta=0.1667 | phase=WARMUP               | val_loss=0.4723 | val_recon=0.2687
  → [WARMUP] beta=0.1667 | val_recon=0.268666 | patience: N/A
  Epoch 006 | beta=0.2000 | phase=WARMUP               | val_loss=0.5086 | val_recon=0.3115
  → [WARMUP] beta=0.2000 | val_recon=0.311485 | patience: N/A
  Epoch 007 | beta=0.2333 | phase=WARMUP               | v

---
## SECTION 6 — Kiểm tra tính hợp lệ sau fix

Điều kiện bắt buộc: `best_epoch > WARMUP_EPOCHS (30)` cho tất cả cấu hình.  
Nếu `best_epoch ≤ 30` → warmup-aware logic chưa hoạt động đúng → cần điều tra thêm.

In [39]:
# ============================================================
# KIỂM TRA TÍNH HỢP LỆ — best_epoch > beta_warmup_epochs
# ------------------------------------------------------------
# Sau khi fix, best_epoch phải nằm trong giai đoạn post-warmup.
# Nếu best_epoch <= WARMUP_EPOCHS thì warmup-aware logic chưa
# hoạt động đúng và cần điều tra thêm.
# ============================================================

print('KIỂM TRA SAU FIX — Warmup-Aware Early Stopping')
print('=' * 70)
print(f'Điều kiện hợp lệ: best_epoch > WARMUP_EPOCHS ({WARMUP_EPOCHS})')
print()
print(f'{"Latent":<10} {"Best Epoch":<15} {"Final Epoch":<15} '
      f'{"Best Val Loss":<18} {"Trạng thái"}')
print('-' * 70)

all_valid = True
for ldim in LATENT_CONFIGS:
    r  = all_results_v2[ldim]
    be = r['best_epoch']
    fe = r['final_epoch']

    # Kiểm tra điều kiện: best_epoch phải nằm sau warmup
    if be is None:
        status    = '✗ best_epoch = None (lỗi!)'
        all_valid = False
    elif be > WARMUP_EPOCHS:
        status = '✓ HỢP LỆ'
    else:
        status    = f'⚠️  WARNING: best_epoch <= warmup={WARMUP_EPOCHS}'
        all_valid = False

    print(f'{ldim:<10} {str(be):<15} {fe:<15} '
          f'{r["best_val_loss"]:<18.6f} {status}')

print('-' * 70)

# ── Thông báo kết quả kiểm tra ─────────────────────────────────
if all_valid:
    print('\n✓ Tất cả best_epoch nằm sau warmup → Fix thành công!')
else:
    print('\n⚠️  Có cấu hình chưa hợp lệ.')
    if QUICK_EXPERIMENT:
        print('   Nguyên nhân có thể: QUICK_EXPERIMENT=True với MAX_EPOCHS quá nhỏ.')
        print('   Đặt QUICK_EXPERIMENT=False để chạy đầy đủ.')

# ============================================================
# THU THẬP METRICS VÀ TẠO DATAFRAME KẾT QUẢ v2
# ============================================================

rows_v2 = []
for ldim in LATENT_CONFIGS:
    r = all_results_v2[ldim]
    m = r['metrics']
    rows_v2.append({
        'latent_dim'   : ldim,
        'n_params'     : r['n_params'],
        'best_epoch'   : r['best_epoch'],
        'final_epoch'  : r['final_epoch'],
        'best_val_loss': round(r['best_val_loss'], 6),
        'threshold'    : round(r['threshold'], 4),
        'accuracy'     : round(m['accuracy'], 4),
        'precision'    : round(m['precision'], 4),
        'recall'       : round(m['recall'], 4),
        'f1_score'     : round(m['f1_score'], 4),
        'roc_auc'      : round(m['roc_auc'], 4) if not np.isnan(m['roc_auc']) else float('nan'),
        'fpr'          : round(m['false_positive_rate'], 4),
        'elapsed_s'    : round(r['elapsed_s'], 1),
        'stopped_early': r['stopped_early'],
    })

df_v2 = pd.DataFrame(rows_v2).set_index('latent_dim')

# ── Lưu comparison_results.csv vào thư mục v2 ──────────────────
csv_v2_path = V2_EXPERIMENTS_DIR / 'comparison_results.csv'
df_v2.to_csv(csv_v2_path)
print(f'\nĐã lưu: {csv_v2_path}')

# ── Hiển thị bảng training stats ────────────────────────────────
print('\nTHONG TIN TRAINING v2:')
print(df_v2[['best_epoch', 'final_epoch', 'best_val_loss', 'stopped_early']].to_string())

KIỂM TRA SAU FIX — Warmup-Aware Early Stopping
Điều kiện hợp lệ: best_epoch > WARMUP_EPOCHS (30)

Latent     Best Epoch      Final Epoch     Best Val Loss      Trạng thái
----------------------------------------------------------------------
8          38              48              0.739679           ✓ HỢP LỆ
16         58              68              0.740856           ✓ HỢP LỆ
32         32              42              0.739344           ✓ HỢP LỆ
----------------------------------------------------------------------

✓ Tất cả best_epoch nằm sau warmup → Fix thành công!

Đã lưu: D:\nids-vae-project\artifacts\experiments\latent_dimension_v2\comparison_results.csv

THONG TIN TRAINING v2:
            best_epoch  final_epoch  best_val_loss  stopped_early
latent_dim                                                       
8                   38           48       0.739679           True
16                  58           68       0.740856           True
32                  32           42   

---
## SECTION 7 — Bảng so sánh kết quả

So sánh đầy đủ `latent_dim ∈ {8, 16, 32}` sau khi fix warmup-aware early stopping.  
Bảng này dùng trực tiếp cho **Chương 4 báo cáo**.

In [40]:
# ============================================================
# BẢNG SO SÁNH METRICS v2 + HIGHLIGHT GIÁ TRỊ TỐT NHẤT
# ------------------------------------------------------------
# So sánh đầy đủ: Accuracy, Precision, Recall, F1, ROC-AUC, FPR
# Highlight: F1 tốt nhất, Recall tốt nhất, ROC-AUC tốt nhất,
#            FPR thấp nhất.
# ============================================================

# ── Tạo bảng hiển thị ──────────────────────────────────────────
display_v2 = df_v2[
    ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'fpr']
].copy()
display_v2.columns  = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'FPR']
display_v2.index.name = 'Latent'

print('BẢNG SO SÁNH HIỆU NĂNG v2 (WARMUP-AWARE EARLY STOPPING)')
print('=' * 72)
print(display_v2.to_string())
print('=' * 72)

# ── Highlight giá trị tốt nhất ─────────────────────────────────
print('\n(*) Highlight — Giá trị tốt nhất:')

# Hàm safe để bỏ qua NaN
def safe_best(series, maximize=True):
    """Trả về (index, value) của giá trị tốt nhất, bỏ qua NaN."""
    valid = series.dropna()
    if valid.empty:
        return None, float('nan')
    idx = valid.idxmax() if maximize else valid.idxmin()
    return idx, valid[idx]

# F1 tốt nhất
f1_idx, f1_val = safe_best(display_v2['F1'])
if f1_idx is not None:
    print(f'  ★ F1 tốt nhất     : latent_dim={f1_idx}  ({f1_val:.4f})')

# Recall tốt nhất
rec_idx, rec_val = safe_best(display_v2['Recall'])
if rec_idx is not None:
    print(f'  ★ Recall tốt nhất : latent_dim={rec_idx}  ({rec_val:.4f})')

# ROC-AUC tốt nhất
roc_idx, roc_val = safe_best(display_v2['ROC-AUC'])
if roc_idx is not None:
    print(f'  ★ ROC-AUC tốt nhất: latent_dim={roc_idx}  ({roc_val:.4f})')

# FPR thấp nhất (lower is better)
fpr_idx, fpr_val = safe_best(display_v2['FPR'], maximize=False)
if fpr_idx is not None:
    print(f'  ★ FPR thấp nhất   : latent_dim={fpr_idx}  ({fpr_val:.4f})')

# ── Bảng đầy đủ với best_epoch để xác nhận fix ─────────────────
print('\nXác nhận best_epoch sau fix (phải > WARMUP_EPOCHS):')
print(df_v2[['best_epoch', 'final_epoch', 'best_val_loss']].to_string())

BẢNG SO SÁNH HIỆU NĂNG v2 (WARMUP-AWARE EARLY STOPPING)
        Accuracy  Precision  Recall      F1  ROC-AUC     FPR
Latent                                                      
8         0.8913     0.8640  0.5748  0.6903   0.7847  0.0242
16        0.8920     0.8654  0.5778  0.6929   0.7835  0.0240
32        0.8903     0.8563  0.5766  0.6891   0.7799  0.0259

(*) Highlight — Giá trị tốt nhất:
  ★ F1 tốt nhất     : latent_dim=16  (0.6929)
  ★ Recall tốt nhất : latent_dim=16  (0.5778)
  ★ ROC-AUC tốt nhất: latent_dim=8  (0.7847)
  ★ FPR thấp nhất   : latent_dim=16  (0.0240)

Xác nhận best_epoch sau fix (phải > WARMUP_EPOCHS):
            best_epoch  final_epoch  best_val_loss
latent_dim                                        
8                   38           48       0.739679
16                  58           68       0.740856
32                  32           42       0.739344


---
## SECTION 8 — Biểu đồ so sánh

- **Biểu đồ 1:** Bar chart so sánh F1 / Recall / FPR (3 metric chính của NIDS)  
- **Biểu đồ 2:** Grouped bar chart so sánh Accuracy / Precision / Recall / F1 / ROC-AUC  

Viền đỏ = giá trị tốt nhất trong nhóm. Biểu đồ lưu tại `artifacts/experiments/latent_dimension_v2/`.

In [43]:
# ============================================================
# SECTION 8 — Biểu đồ so sánh latent dimension (v2)
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display as ipy_display

# --- Dữ liệu từ df_v2 (index = latent_dim) ---
latent_dims = list(df_v2.index)                          # [8, 16, 32]
f1_scores   = list(df_v2['f1_score'].astype(float))      # cột f1_score
recalls     = list(df_v2['recall'].astype(float))
fprs        = list(df_v2['fpr'].astype(float))
accs        = list(df_v2['accuracy'].astype(float))
precs       = list(df_v2['precision'].astype(float))
aucs        = list(df_v2['roc_auc'].astype(float))

labels = [f"latent={d}" for d in latent_dims]
x = np.arange(len(labels))

# Helper: trả về danh sách màu viền (đỏ = best)
def edge_colors_viz(values, lower_is_better=False):
    best_idx = int(np.argmin(values) if lower_is_better else np.argmax(values))
    return ['#d62728' if i == best_idx else '#888888' for i in range(len(values))]

# ============================================================
# Biểu đồ 1 — F1 / Recall / FPR (3 metric chính NIDS)
# ============================================================
fig1, axes1 = plt.subplots(1, 3, figsize=(13, 4))
fig1.suptitle("Latent Dimension Comparison (v2) — Key NIDS Metrics", fontsize=13, fontweight='bold')

metrics1 = [
    ("F1-Score",    f1_scores,  False, '#1f77b4'),
    ("Recall",      recalls,    False, '#2ca02c'),
    ("FPR",         fprs,       True,  '#ff7f0e'),
]

for ax, (title, vals, lower, color) in zip(axes1, metrics1):
    ec = edge_colors_viz(vals, lower_is_better=lower)
    bars = ax.bar(labels, vals, color=color, alpha=0.75, edgecolor=ec, linewidth=2.2)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_ylabel(title)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
                f"{v:.4f}", ha='center', va='bottom', fontsize=9)
    best_label = "(↓ best)" if lower else "(↑ best)"
    ax.set_xlabel(f"Viền đỏ = best {best_label}", fontsize=8, color='#d62728')

fig1.tight_layout()
out1 = V2_EXPERIMENTS_DIR / 'latent_comparison_v2_key_metrics.png'
fig1.savefig(out1, dpi=150, bbox_inches='tight')
print(f"✅ Biểu đồ 1 lưu: {out1}")
ipy_display(fig1)
plt.close(fig1)

# ============================================================
# Biểu đồ 2 — Grouped bar chart 5 metrics
# ============================================================
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
metric_data  = [accs, precs, recalls, f1_scores, aucs]

n_metrics = len(metric_names)
n_dims    = len(latent_dims)
bar_width = 0.18
offsets   = np.linspace(-(n_dims - 1) / 2 * bar_width, (n_dims - 1) / 2 * bar_width, n_dims)
colors    = ['#1f77b4', '#2ca02c', '#ff7f0e']

fig2, ax2 = plt.subplots(figsize=(13, 5))
ax2.set_title("Latent Dimension Comparison (v2) — All Metrics", fontsize=13, fontweight='bold')

xi = np.arange(n_metrics)
for dim_i, (dim, offset, color) in enumerate(zip(latent_dims, offsets, colors)):
    vals_dim = [metric_data[m][dim_i] for m in range(n_metrics)]
    bars = ax2.bar(xi + offset, vals_dim, width=bar_width, label=f"latent={dim}",
                   color=color, alpha=0.75, edgecolor='#333333', linewidth=0.8)
    for bar, v in zip(bars, vals_dim):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f"{v:.3f}", ha='center', va='bottom', fontsize=7.5)

ax2.set_xticks(xi)
ax2.set_xticklabels(metric_names, fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.set_ylim(0, 1.15)
ax2.legend(title="Latent dim", fontsize=10)
ax2.grid(axis='y', alpha=0.3)

fig2.tight_layout()
out2 = V2_EXPERIMENTS_DIR / 'latent_comparison_v2_grouped.png'
fig2.savefig(out2, dpi=150, bbox_inches='tight')
print(f"✅ Biểu đồ 2 lưu: {out2}")
ipy_display(fig2)
plt.close(fig2)

✅ Biểu đồ 1 lưu: D:\nids-vae-project\artifacts\experiments\latent_dimension_v2\latent_comparison_v2_key_metrics.png


<Figure size 1300x400 with 3 Axes>

✅ Biểu đồ 2 lưu: D:\nids-vae-project\artifacts\experiments\latent_dimension_v2\latent_comparison_v2_grouped.png


<Figure size 1300x500 with 1 Axes>

---
## SECTION 9 — Kết luận thí nghiệm v2

Sinh phần nhận xét hoàn chỉnh: root cause lỗi v1, logic fix, best epoch sau fix, latent dimension tốt nhất.

In [41]:
# ============================================================
# TỰ ĐỘNG SINH KẾT LUẬN DỰA TRÊN KẾT QUẢ THỰC TẾ v2
# ------------------------------------------------------------
# Tất cả số liệu được lấy từ all_results_v2 / df_v2.
# Không hardcode bất kỳ giá trị nào.
# ============================================================

def fmt_v2(val, decimals=4):
    """Định dạng số thực, thay NaN bằng 'N/A'."""
    if isinstance(val, float) and np.isnan(val):
        return 'N/A'
    return f'{val:.{decimals}f}'

# ── Lấy giá trị từ df_v2 ────────────────────────────────────────
be8  = all_results_v2[8]['best_epoch']
be16 = all_results_v2[16]['best_epoch']
be32 = all_results_v2[32]['best_epoch']
fe8  = all_results_v2[8]['final_epoch']
fe16 = all_results_v2[16]['final_epoch']
fe32 = all_results_v2[32]['final_epoch']

# Tìm cấu hình tốt nhất theo từng tiêu chí
best_f1_dim_v2,  best_f1_val_v2  = safe_best(df_v2['f1_score'])
best_roc_dim_v2, best_roc_val_v2 = safe_best(df_v2['roc_auc'])
best_rec_dim_v2, best_rec_val_v2 = safe_best(df_v2['recall'])
low_fpr_dim_v2,  low_fpr_val_v2  = safe_best(df_v2['fpr'], maximize=False)

# ── Sinh bảng Markdown cho báo cáo ─────────────────────────────
def fmt_row_v2(ldim):
    """Tạo một dòng bảng Markdown cho latent_dim chỉ định."""
    r = df_v2.loc[ldim]
    return (
        f'| {ldim:<6} | {fmt_v2(r["accuracy"])} | {fmt_v2(r["precision"])} | '
        f'{fmt_v2(r["recall"])} | {fmt_v2(r["f1_score"])} | '
        f'{fmt_v2(r["roc_auc"])} | {fmt_v2(r["fpr"])} |'
    )

table_md_v2 = (
    '| Latent | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR |\n'
    '|:------:|:--------:|:---------:|:------:|:--:|:-------:|:---:|\n'
    + '\n'.join(fmt_row_v2(d) for d in LATENT_CONFIGS)
)

# ── Sinh văn bản kết luận ────────────────────────────────────────
conclusion_v2 = f"""
{'━' * 70}
KẾT LUẬN THÍ NGHIỆM LATENT DIMENSION — v2 (WARMUP-AWARE)
{'━' * 70}

QUICK_EXPERIMENT = {QUICK_EXPERIMENT}
{'⚠️  Quick mode KHÔNG dùng cho báo cáo chính thức.' if QUICK_EXPERIMENT else '✓ Full mode — kết quả có thể dùng cho báo cáo.'}

────────────────────────────────────────────────────────────────────
1. KẾT QUẢ CŨ (v1) SAI Ở ĐÂU?
────────────────────────────────────────────────────────────────────
- Code v1 so sánh total val_loss = recon + beta * kl để chọn best_epoch.
- Trong warmup, beta tăng dần từ 0 → 1 nên val_loss tự nhiên tăng mỗi epoch
  dù model không hề xấu đi.
- Early stopping nhầm lẫn đây là "không cải thiện" và kích hoạt sớm:
    best_epoch  = 1    (epoch đầu tiên có beta nhỏ nhất → val_loss thấp nhất)
    final_epoch = 11   (= 1 + patience = 1 + 10)
- Model dừng hoàn toàn trong giai đoạn warmup, chưa bao giờ học đúng nghĩa.

────────────────────────────────────────────────────────────────────
2. LOGIC WARMUP-AWARE ĐÃ SỬA GÌ?
────────────────────────────────────────────────────────────────────
- PHASE 1 (WARMUP, epoch 1 → {WARMUP_EPOCHS}):
  * KHÔNG đếm epochs_no_improve.
  * Checkpoint theo val_recon (ổn định, không phụ thuộc beta).
- PHASE 2 (epoch {WARMUP_EPOCHS + 1} — POST_WARMUP_START):
  * Reset: best_val_loss = ∞, epochs_no_improve = 0, best_epoch = None.
  * Bắt đầu monitor total val_loss với beta đã cố định = {BETA_END}.
- PHASE 3 (epoch > {WARMUP_EPOCHS} — POST_WARMUP):
  * Early stopping bình thường theo total val_loss.
  * Val_loss phản ánh đúng chất lượng model vì beta không còn thay đổi.

────────────────────────────────────────────────────────────────────
3. BEST EPOCH SAU FIX
────────────────────────────────────────────────────────────────────
  latent_dim = 8  : best_epoch = {be8}  | final_epoch = {fe8}
  latent_dim = 16 : best_epoch = {be16}  | final_epoch = {fe16}
  latent_dim = 32 : best_epoch = {be32}  | final_epoch = {fe32}
  (Tất cả best_epoch phải > {WARMUP_EPOCHS} — xác nhận ở Section 10)

────────────────────────────────────────────────────────────────────
4. LATENT DIMENSION TỐT NHẤT (v2)
────────────────────────────────────────────────────────────────────
  Theo F1-Score  : latent_dim = {best_f1_dim_v2}   ({fmt_v2(best_f1_val_v2)})
  Theo Recall    : latent_dim = {best_rec_dim_v2}   ({fmt_v2(best_rec_val_v2)})
  Theo ROC-AUC   : latent_dim = {best_roc_dim_v2}   ({fmt_v2(best_roc_val_v2)})
  Theo FPR (↓)   : latent_dim = {low_fpr_dim_v2}   ({fmt_v2(low_fpr_val_v2)})

────────────────────────────────────────────────────────────────────
5. CÓ THỂ ĐƯA VÀO BÁO CÁO KHÔNG?
────────────────────────────────────────────────────────────────────
{'  ✓ QUICK_EXPERIMENT=False — kết quả đủ tin cậy cho báo cáo.' if not QUICK_EXPERIMENT else '  ✗ QUICK_EXPERIMENT=True — CẦN CHẠY LẠI với QUICK_EXPERIMENT=False.'}

Bảng kết quả (copy vào Chương 4):
{table_md_v2}

Artifacts lưu tại: artifacts/experiments/latent_dimension_v2/
{'━' * 70}
"""

print(conclusion_v2)

# ── Lưu kết luận ra file txt ────────────────────────────────────
report_v2_path = V2_EXPERIMENTS_DIR / 'report_section_4x.txt'
with open(report_v2_path, 'w', encoding='utf-8') as fh:
    fh.write(conclusion_v2)
print(f'Đã lưu kết luận: {report_v2_path}')

# ── Liệt kê tất cả files đã tạo (v2) ────────────────────────────
print('\nCÁC FILE ĐÃ TẠO (v2):')
print('=' * 65)
files_v2 = []
for ldim in LATENT_CONFIGS:
    d = V2_EXPERIMENTS_DIR / f'latent_{ldim}'
    for fname in [
        'model_config.json', 'training_history.json', 'training_summary.json',
        'threshold.json', 'evaluation_metrics.json', 'vae_best.pth',
    ]:
        files_v2.append(d / fname)
files_v2 += [
    V2_EXPERIMENTS_DIR / 'comparison_results.csv',
    V2_EXPERIMENTS_DIR / 'report_section_4x.txt',
]

for fp in files_v2:
    status_icon = '✓' if fp.exists() else '✗ (chưa tạo)'
    try:
        rel = fp.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = fp
    print(f'  {status_icon}  {rel}')

print('=' * 65)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KẾT LUẬN THÍ NGHIỆM LATENT DIMENSION — v2 (WARMUP-AWARE)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

QUICK_EXPERIMENT = False
✓ Full mode — kết quả có thể dùng cho báo cáo.

────────────────────────────────────────────────────────────────────
1. KẾT QUẢ CŨ (v1) SAI Ở ĐÂU?
────────────────────────────────────────────────────────────────────
- Code v1 so sánh total val_loss = recon + beta * kl để chọn best_epoch.
- Trong warmup, beta tăng dần từ 0 → 1 nên val_loss tự nhiên tăng mỗi epoch
  dù model không hề xấu đi.
- Early stopping nhầm lẫn đây là "không cải thiện" và kích hoạt sớm:
    best_epoch  = 1    (epoch đầu tiên có beta nhỏ nhất → val_loss thấp nhất)
    final_epoch = 11   (= 1 + patience = 1 + 10)
- Model dừng hoàn toàn trong giai đoạn warmup, chưa bao giờ học đúng nghĩa.

────────────────────────────────────────────────────────────────────
2. LOGIC WARMUP-AWARE ĐÃ SỬA GÌ?
──────